
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# Demo - Clean, Transform, and Chunk Parsed Text

## Overview
In this demo, we’ll learn how to **clean** and **transform** parsed document text for effective use with language models, and then **chunk** the text for retrieval workflows. The parsed text is currently in JSON format, and we’ll demonstrate two methods to convert it into plain, clean text:

## Learning Objectives
By the end of this demo, you will be able to:
1. **Transform** parsed JSON text into clean, markdown-formatted or plain text suitable for LLMs.
2. **Compare** two transformation methods: LLM-powered semantic cleaning and fast concatenation.
3. **Chunk** the cleaned text by page, with overlap for context, using LangChain.
4. **Store** the final chunked table for downstream embedding and AI Search.

## Requirements
* **Parsed document table** in JSON format. This table is created in the previous demo. If you haven't done so yet, **you must complete this demo first (`2.2 Demo - Parse Documents to Structured Data`)**.
* **Serverless Compute (environment version 5)**. Follow the instructions [here](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) to select the appropriate environment version.
* Required libraries are added to **Dependencies** of Serverless compute configuration.

## Setup

Run the code below to configure the classroom environment. This step ensures all dependencies are available and your workspace is ready for the demo.

In [0]:
%run ../Includes/Classroom-Setup-02

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


**Set default catalog and schema:**

Default catalog and schema are set.


**Prepare Datasets:**

✅ Volume orion_docs already exists. Skipping copy.
✅ Volume orion_text already exists. Skipping copy.


**Checking for right compute environment:**

✅ Environment check passed: Serverless 5 detected.


**Variables needed for this demo/lab:**

✍️ Sample documents volume: dbacademy.labuser15785576_1783348796.orion_docs


## A. Transform Parsed JSON to Clean Text

In this section, we’ll convert the parsed JSON text into clean, plain text that’s ready for use with language models. We’ll demonstrate two methods:

1. **LLM-powered semantic cleaning** using `ai_query` to batch process and convert JSON to markdown text. This method preserves more document semantics but may be more expensive.
2. **Fast concatenation** by joining all text elements into a single plain text string. This method is quick and cost-effective, but loses some semantic structure (e.g., page headers).

*In both methods, we’ll use a `== page ==` token to separate pages for later chunking and retrieval workflows.*

### A1. Load Parsed Document

Let’s begin by loading the parsed documents generated in the previous demo. This step ensures we have the structured data needed for cleaning and transformation.

*Reminder: Make sure the parsed table exists and is up to date before proceeding.*

In [0]:
parsed_table = f"{catalog}.{schema}.docs_parsed"
chunked_table = f"{catalog}.{schema}.docs_chunked"

parsed_df = spark.read.table(parsed_table)

print(f"Loaded parsed documents from: {parsed_table}")
parsed_df.printSchema()

Loaded parsed documents from: dbacademy.labuser15785576_1783348796.docs_parsed
root
 |-- path: string (nullable = true)
 |-- modificationTime: timestamp (nullable = true)
 |-- length: long (nullable = true)
 |-- parsed_content: variant (nullable = true)



### A2. LLM-Powered Semantic Cleaning with ai_query

In this method, we use the **`ai_query` AI function** to batch process the parsed JSON text and convert it into clean, markdown-formatted text. This approach leverages a large language model (LLM) to preserve document semantics, such as headers, tables, and structure, making the output more useful for downstream LLM tasks.

- **Pros:** Retains more structure and meaning, produces high-quality markdown.
- **Cons:** May be more expensive due to LLM usage.

**⚠️ Warning**: LLM-powered cleaning may incur higher costs for processing many documents. Use fast concatenation for quick, low-cost processing.

**Note:** The `ai_query` AI function supports many foundation models, including those from OpenAI, Anthropic, and Databricks. In this demo, we will use Anthropic's Claude Sonnet 4.6 model. Claude models return plain text by default; do not pass `responseFormat => '{"type":"text"}'`, as that format is not supported.

We’ll instruct the LLM to parse the JSON and output clean markdown, using `== page ==` as a separator between pages.

In [0]:
from pyspark.sql.functions import expr

display(
    parsed_df.select(
        expr(f"""
            CONCAT('<some prompt> with data:', CAST(parsed_content AS STRING))
        """
        )
    )
)

concat( with data:, CAST(parsed_content AS STRING)) with data:{"document":{"elements":[{"bbox":[{"coord":[383,142,675,177],"page_id":0}],"confidence":0.9995,"content":"Orion Robotics","description":null,"id":0,"type":"title"},{"bbox":[{"coord":[261,252,797,281],"page_id":0}],"confidence":0.9957,"content":"Orion A1 – Battery Management System Manual","description":null,"id":1,"type":"title"},{"bbox":[{"coord":[328,302,731,322],"page_id":0}],"confidence":0.9842,"content":"© 2025 Orion Robotics – Internal Engineering Reference","description":null,"id":2,"type":"text"},{"bbox":[{"coord":[116,354,917,463],"page_id":0}],"confidence":0.9873,"content":"This manual provides an in-depth overview of the Orion A1 Battery Management System (BMS), covering hardware design, safety mechanisms, and maintenance best practices. The BMS is a critical subsystem responsible for monitoring, protecting, and optimizing the lithium-ion battery modules powering the Orion A1 platform.","description":null,"id":3,"type":"text"},{"bbox":[{"coord":[117,490,941,598],"page_id":0}],"confidence":0.9893,"content":"The Orion BMS ensures long-term battery health by dynamically balancing cell voltages,\nregulating current flow, and monitoring thermal conditions through embedded temperature\nsensors. The system's adaptive charge algorithm was developed in collaboration with the Orion\nPower Systems division, achieving a balance between performance and safety.","description":null,"id":4,"type":"text"},{"bbox":[{"coord":[117,100,334,129],"page_id":1}],"confidence":0.9997,"content":"1. System Architecture","description":null,"id":5,"type":"section_header"},{"bbox":[{"coord":[115,142,925,282],"page_id":1}],"confidence":0.9919,"content":"The BMS consists of two primary components: the Master Control Unit (MCU) and distributed\nCell Monitoring Boards (CMBs). The MCU consolidates telemetry from each CMB, computes\ncharge/discharge limits, and communicates with the motion controller via the OrionBus CAN\nprotocol. Each CMB continuously tracks cell voltage (±1mV accuracy), temperature, and\nimpedance.","description":null,"id":6,"type":"text"},{"bbox":[{"coord":[115,301,942,438],"page_id":1}],"confidence":0.9909,"content":"Thermal management is handled by a closed-loop liquid cooling system, which is activated\nautomatically when the cell temperature exceeds 38°C. The system's safety layer includes dual\novercurrent protection and hardware-based short-circuit isolation. If a thermal runaway\ncondition is detected, the BMS triggers a controlled shutdown sequence within 200\nmilliseconds.","description":null,"id":7,"type":"text"},{"bbox":[{"coord":[216,474,818,764],"page_id":1}],"confidence":0.9228,"content":"Battery Efficiency vs Temperature\nEfficiency (%)\n87.5\n90.0\n92.5\n95.0\n0\n10\n20\n30\n40\n50\nTemperature (°C)","description":"A line graph plots battery efficiency in percentage against temperature in degrees Celsius, showing a peak around 30 degrees.","id":8,"type":"figure"},{"bbox":[{"coord":[270,780,788,802],"page_id":1}],"confidence":0.9876,"content":"Figure 1. Efficiency of Orion A1 battery pack across temperature ranges.","description":null,"id":9,"type":"caption"},{"bbox":[{"coord":[115,100,309,127],"page_id":2}],"confidence":0.9998,"content":"2. Safety Guidelines","description":null,"id":10,"type":"section_header"},{"bbox":[{"coord":[115,142,943,281],"page_id":2}],"confidence":0.9756,"content":"Field engineers must adhere to strict handling procedures when working on battery modules.\nEach pack must be isolated from the main bus before removal. The Orion Diagnostic Interface (ODI) should be used to verify zero current before disengagement. Never expose the battery to open flame, solvents, or humidity above 90%. Only Orion-certified technicians are authorized to\nperform module replacements.","description":null,"id":11,"type":"text"},{"bbox":[{"coord":[241,311,814,601],"page_id":2}],"confidence":0.925,"content":"Charge Retention Over Life Cycle\nCapacity Retaine

In [0]:
from pyspark.sql.functions import expr

# Choose a Databricks foundation model (or your own serving endpoint name)
ENDPOINT = "databricks-claude-sonnet-4-6"

# Example prompt for the LLM
prompt_prefix = '''
You are a helpful assistant. Given a JSON object representing a parsed document (with pages, elements, and metadata), convert the content into clean, readable markdown. Use "== page ==" to separate each page. Preserve important structure such as headers, tables, and captions. Do not include any JSON or code blocks in the output—just the clean markdown text.

JSON:

'''

# Apply ai_query to batch process the parsed JSON text
# Note: Claude models do not support responseFormat type "text"; omit it for plain-text output.
transformed_df = (
    parsed_df.withColumn(
        "clean_markdown_text",
        expr(f"""
          ai_query(
            '{ENDPOINT}',
            CONCAT('{prompt_prefix}', CAST(parsed_content AS STRING))
          )
        """)
    )
)

display(transformed_df.select("path", "clean_markdown_text"))

path clean_markdown_text dbfs:/Volumes/dbacademy/labuser15785576_1783348796/orion_docs/02_Orion_Battery_Management_System_Manual.pdf # Orion Robotics

## Orion A1 – Battery Management System Manual

© 2025 Orion Robotics – Internal Engineering Reference

This manual provides an in-depth overview of the Orion A1 Battery Management System (BMS), covering hardware design, safety mechanisms, and maintenance best practices. The BMS is a critical subsystem responsible for monitoring, protecting, and optimizing the lithium-ion battery modules powering the Orion A1 platform.

The Orion BMS ensures long-term battery health by dynamically balancing cell voltages, regulating current flow, and monitoring thermal conditions through embedded temperature sensors. The system's adaptive charge algorithm was developed in collaboration with the Orion Power Systems division, achieving a balance between performance and safety.

== page ==

## 1. System Architecture

The BMS consists of two primary components: the Master Control Unit (MCU) and distributed Cell Monitoring Boards (CMBs). The MCU consolidates telemetry from each CMB, computes charge/discharge limits, and communicates with the motion controller via the OrionBus CAN protocol. Each CMB continuously tracks cell voltage (±1mV accuracy), temperature, and impedance.

Thermal management is handled by a closed-loop liquid cooling system, which is activated automatically when the cell temperature exceeds 38°C. The system's safety layer includes dual overcurrent protection and hardware-based short-circuit isolation. If a thermal runaway condition is detected, the BMS triggers a controlled shutdown sequence within 200 milliseconds.

![Battery Efficiency vs Temperature — A line graph plots battery efficiency in percentage against temperature in degrees Celsius, showing a peak around 30 degrees.](figure1)

**Figure 1.** Efficiency of Orion A1 battery pack across temperature ranges.

== page ==

## 2. Safety Guidelines

Field engineers must adhere to strict handling procedures when working on battery modules. Each pack must be isolated from the main bus before removal. The Orion Diagnostic Interface (ODI) should be used to verify zero current before disengagement. Never expose the battery to open flame, solvents, or humidity above 90%. Only Orion-certified technicians are authorized to perform module replacements.

![Charge Retention Over Life Cycle — A line graph displays a decreasing trend of charge retention percentage against charge cycles, ranging from 0 to 1000.](figure2)

**Figure 2.** Retained charge capacity over lifecycle testing.

## 3. Technical Specifications

| Attribute | Specification |
|---|---|
| Nominal Voltage | 52.8 V |
| Capacity | 18 Ah |
| Energy Density | 245 Wh/kg |
| Cycle Life (80%) | 1000 cycles |
| Operating Temp Range | 0–45 °C |
| Protection | Dual redundant current sensors + fuse isolation |

Routine inspection of the battery system is recommended every 500 operational hours. Key diagnostic parameters include cell voltage deviation (<0.02V), temperature differential (<3°C), and charge retention after standby. Logs generated by the BMS can be exported via the Orion Diagnostics Utility for advanced analysis or remote monitoring.

---
*End of Document* dbfs:/Volumes/dbacademy/labuser15785576_1783348796/orion_docs/01_Orion_A1_Product_Overview.pdf # Orion Robotics

Orion A1 Product Overview

© 2025 Orion Robotics – Confidential

The Orion A1 humanoid service robot is the flagship model in the Orion Robotics product line. Designed for adaptive environments, A1 combines modular hardware, precision motion control, and deep learning-based sensory perception to perform tasks ranging from hospitality assistance to warehouse logistics.

Since its initial release in 2023, Orion A1 has been deployed in over 40 countries. Each production cycle incorporates firmware refinements and mechanical upgrades based on telemetry data collected from field units. The 2025 A1 model introduces a

### A3. Fast Plain Text Conversion

In this method, we rapidly concatenate all text elements from the parsed JSON into a single plain text string. This approach is fast and cost-effective, but it sacrifices document structure—such as headers, tables, and captions.

- **Pros:** Fast, inexpensive, and simple to implement.
- **Cons:** Loses important structure and semantics.

**NOTE:** We use Spark to extract and join text from each page, inserting a `== page ==` token between pages for later chunking.

The content extraction logic is provided in the `Includes/content_extractor` file.

In [0]:
from pyspark.sql import functions as F

# Convert VARIANT/struct/map to a JSON string first (avoids VariantVal issues)
safe_json_col = F.coalesce(
    F.to_json(F.col("parsed_content")),
    F.col("parsed_content").cast("string")
)

# Apply the UDF
plain_text_df = parsed_df.withColumn(
    "plain_text",
    extract_contents_udf()(safe_json_col)
)

display(plain_text_df.select("path", "plain_text"))


path plain_text dbfs:/Volumes/dbacademy/labuser15785576_1783348796/orion_docs/02_Orion_Battery_Management_System_Manual.pdf Orion Robotics

Orion A1 – Battery Management System Manual

© 2025 Orion Robotics – Internal Engineering Reference
This manual provides an in-depth overview of the Orion A1 Battery Management System (BMS), covering hardware design, safety mechanisms, and maintenance best practices. The BMS is a critical subsystem responsible for monitoring, protecting, and optimizing the lithium-ion battery modules powering the Orion A1 platform.
The Orion BMS ensures long-term battery health by dynamically balancing cell voltages,
regulating current flow, and monitoring thermal conditions through embedded temperature
sensors. The system's adaptive charge algorithm was developed in collaboration with the Orion
Power Systems division, achieving a balance between performance and safety.

== page ==

1. System Architecture

The BMS consists of two primary components: the Master Control Unit (MCU) and distributed
Cell Monitoring Boards (CMBs). The MCU consolidates telemetry from each CMB, computes
charge/discharge limits, and communicates with the motion controller via the OrionBus CAN
protocol. Each CMB continuously tracks cell voltage (±1mV accuracy), temperature, and
impedance.
Thermal management is handled by a closed-loop liquid cooling system, which is activated
automatically when the cell temperature exceeds 38°C. The system's safety layer includes dual
overcurrent protection and hardware-based short-circuit isolation. If a thermal runaway
condition is detected, the BMS triggers a controlled shutdown sequence within 200
milliseconds.
Battery Efficiency vs Temperature
Efficiency (%)
87.5
90.0
92.5
95.0
0
10
20
30
40
50
Temperature (°C)

Figure 1. Efficiency of Orion A1 battery pack across temperature ranges.


== page ==

2. Safety Guidelines

Field engineers must adhere to strict handling procedures when working on battery modules.
Each pack must be isolated from the main bus before removal. The Orion Diagnostic Interface (ODI) should be used to verify zero current before disengagement. Never expose the battery to open flame, solvents, or humidity above 90%. Only Orion-certified technicians are authorized to
perform module replacements.
Charge Retention Over Life Cycle
Capacity Retained (%)
0
200
400
600
800
1000
Charge Cycles

Figure 2. Retained charge capacity over lifecycle testing.

3. Technical Specifications

 Attribute Specification Nominal Voltage 52.8 V Capacity 18 Ah Energy Density 245 Wh/kg Cycle Life (80%) 1000 cycles Operating Temp Range 0–45 °C Protection Dual redundant current sensors + fuse isolation 

Routine inspection of the battery system is recommended every 500 operational hours. Key
diagnostic parameters include cell voltage deviation (<0.02V), temperature differential (<3°C),
and charge retention after standby. Logs generated by the BMS can be exported via the Orion
Diagnostics Utility for advanced analysis or remote monitoring.
End of Document
 dbfs:/Volumes/dbacademy/labuser15785576_1783348796/orion_docs/01_Orion_A1_Product_Overview.pdf Orion Robotics

Orion A1 Product Overview
© 2025 Orion Robotics – Confidential
The Orion A1 humanoid service robot is the flagship model in the Orion Robotics product line.
Designed for adaptive environments, A1 combines modular hardware, precision motion control,
and deep learning-based sensory perception to perform tasks ranging from hospitality
assistance to warehouse logistics.
Since its initial release in 2023, Orion A1 has been deployed in over 40 countries. Each production cycle incorporates firmware refinements and mechanical upgrades based on telemetry data collected from field units. The 2025 A1 model introduces an improved gait controller and battery module rated for 10 hours of continuous operation.

== page ==

1. System Architecture

The Orion A1 platform is built on a modular architecture comprising four key subsystems:
motion, vision, cognition

## B. Chunk Cleaned Text for Retrieval

Now that we have clean, page-separated text, we’ll chunk it for retrieval workflows. Chunking helps language models and AI Search systems efficiently process and retrieve relevant information.

We’ll use LangChain’s `RecursiveCharacterTextSplitter` to split the text by the `== page ==` token. This utility automatically handles chunking and overlap, making it easy to prepare text for embedding and search.

**Note:** Overlap is only introduced when a single input is split into multiple chunks. In this example, each page typically forms one chunk with `chunk_size=2000`, so some rows may not show any overlap. To observe overlapping chunks more clearly, try reducing the chunk size.

In [0]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pyspark.sql.types import StructType, StructField, StringType
import pandas as pd

# Set chunking parameters: chunk_size controls the max length of each chunk, and chunk_overlap allows for overlapping text between chunks to improve retrieval quality.
CHUNK_SIZE = 2000
CHUNK_OVERLAP = 200

# Build the text splitter with preferred separators.
# This splitter will break the text at page markers or other natural boundaries, preserving document structure where possible.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n== page ==\n", "== page ==", "\n\n", "\n", " ", ""]
)

# Define the output schema for the chunked DataFrame.
# Each row will contain the document path and a single text chunk.
schema = StructType([
    StructField("path", StringType(), True),
    StructField("chunk", StringType(), True),
])

def split_rows(iterator):
    """
    mapInPandas function: input pdfs with columns [path, plain_text],
    output rows [path, chunk].
    This function splits each document's text into chunks and yields them for DataFrame construction.
    """
    for pdf in iterator:
        out = []
        for _, row in pdf.iterrows():
            path = row["path"]
            text = row["plain_text"]
            if isinstance(text, str) and text.strip():
                for c in splitter.split_text(text):
                    if c and c.strip():
                        out.append((path, c))
        yield pd.DataFrame(out, columns=["path", "chunk"])

# Apply the splitter to the plain text DataFrame.
# This step transforms each document into multiple chunked rows for efficient downstream retrieval and embedding.
df_chunks = (
    plain_text_df
    .select("path", "plain_text")
    .mapInPandas(split_rows, schema=schema)
)

# Display the resulting chunked DataFrame for inspection.
display(df_chunks)

path chunk dbfs:/Volumes/dbacademy/labuser15785576_1783348796/orion_docs/02_Orion_Battery_Management_System_Manual.pdf Orion Robotics

Orion A1 – Battery Management System Manual

© 2025 Orion Robotics – Internal Engineering Reference
This manual provides an in-depth overview of the Orion A1 Battery Management System (BMS), covering hardware design, safety mechanisms, and maintenance best practices. The BMS is a critical subsystem responsible for monitoring, protecting, and optimizing the lithium-ion battery modules powering the Orion A1 platform.
The Orion BMS ensures long-term battery health by dynamically balancing cell voltages,
regulating current flow, and monitoring thermal conditions through embedded temperature
sensors. The system's adaptive charge algorithm was developed in collaboration with the Orion
Power Systems division, achieving a balance between performance and safety.

== page ==

1. System Architecture

The BMS consists of two primary components: the Master Control Unit (MCU) and distributed
Cell Monitoring Boards (CMBs). The MCU consolidates telemetry from each CMB, computes
charge/discharge limits, and communicates with the motion controller via the OrionBus CAN
protocol. Each CMB continuously tracks cell voltage (±1mV accuracy), temperature, and
impedance.
Thermal management is handled by a closed-loop liquid cooling system, which is activated
automatically when the cell temperature exceeds 38°C. The system's safety layer includes dual
overcurrent protection and hardware-based short-circuit isolation. If a thermal runaway
condition is detected, the BMS triggers a controlled shutdown sequence within 200
milliseconds.
Battery Efficiency vs Temperature
Efficiency (%)
87.5
90.0
92.5
95.0
0
10
20
30
40
50
Temperature (°C)

Figure 1. Efficiency of Orion A1 battery pack across temperature ranges. dbfs:/Volumes/dbacademy/labuser15785576_1783348796/orion_docs/02_Orion_Battery_Management_System_Manual.pdf == page ==

2. Safety Guidelines

Field engineers must adhere to strict handling procedures when working on battery modules.
Each pack must be isolated from the main bus before removal. The Orion Diagnostic Interface (ODI) should be used to verify zero current before disengagement. Never expose the battery to open flame, solvents, or humidity above 90%. Only Orion-certified technicians are authorized to
perform module replacements.
Charge Retention Over Life Cycle
Capacity Retained (%)
0
200
400
600
800
1000
Charge Cycles

Figure 2. Retained charge capacity over lifecycle testing.

3. Technical Specifications

 Attribute Specification Nominal Voltage 52.8 V Capacity 18 Ah Energy Density 245 Wh/kg Cycle Life (80%) 1000 cycles Operating Temp Range 0–45 °C Protection Dual redundant current sensors + fuse isolation 

Routine inspection of the battery system is recommended every 500 operational hours. Key
diagnostic parameters include cell voltage deviation (<0.02V), temperature differential (<3°C),
and charge retention after standby. Logs generated by the BMS can be exported via the Orion
Diagnostics Utility for advanced analysis or remote monitoring.
End of Document dbfs:/Volumes/dbacademy/labuser15785576_1783348796/orion_docs/01_Orion_A1_Product_Overview.pdf Orion Robotics

Orion A1 Product Overview
© 2025 Orion Robotics – Confidential
The Orion A1 humanoid service robot is the flagship model in the Orion Robotics product line.
Designed for adaptive environments, A1 combines modular hardware, precision motion control,
and deep learning-based sensory perception to perform tasks ranging from hospitality
assistance to warehouse logistics.
Since its initial release in 2023, Orion A1 has been deployed in over 40 countries. Each production cycle incorporates firmware refinements and mechanical upgrades based on telemetry data collected from field units. The 2025 A1 model introduces an improved gait controller and battery module rated for 10 hours of continuous operation. dbfs:/Volumes/dbacademy/labuser15785576_1783348796/

## C. Save Chunked Data to Delta Table

Let’s save our chunked text data to a Delta table for downstream embedding and retrieval workflows.

**Task:** Write the chunked DataFrame to the table defined as **`chunked_table`** in the setup section.

In [0]:
from pyspark.sql import functions as F

# Add a unique, incremental id column before saving
df_chunks = df_chunks.withColumn("id", F.monotonically_increasing_id())

# Save the chunked data with id to the Delta table for retrieval and embedding
df_chunks.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(chunked_table)

display(spark.read.table(chunked_table))

path chunk id dbfs:/Volumes/dbacademy/labuser15785576_1783348796/orion_docs/02_Orion_Battery_Management_System_Manual.pdf Orion Robotics

Orion A1 – Battery Management System Manual

© 2025 Orion Robotics – Internal Engineering Reference
This manual provides an in-depth overview of the Orion A1 Battery Management System (BMS), covering hardware design, safety mechanisms, and maintenance best practices. The BMS is a critical subsystem responsible for monitoring, protecting, and optimizing the lithium-ion battery modules powering the Orion A1 platform.
The Orion BMS ensures long-term battery health by dynamically balancing cell voltages,
regulating current flow, and monitoring thermal conditions through embedded temperature
sensors. The system's adaptive charge algorithm was developed in collaboration with the Orion
Power Systems division, achieving a balance between performance and safety.

== page ==

1. System Architecture

The BMS consists of two primary components: the Master Control Unit (MCU) and distributed
Cell Monitoring Boards (CMBs). The MCU consolidates telemetry from each CMB, computes
charge/discharge limits, and communicates with the motion controller via the OrionBus CAN
protocol. Each CMB continuously tracks cell voltage (±1mV accuracy), temperature, and
impedance.
Thermal management is handled by a closed-loop liquid cooling system, which is activated
automatically when the cell temperature exceeds 38°C. The system's safety layer includes dual
overcurrent protection and hardware-based short-circuit isolation. If a thermal runaway
condition is detected, the BMS triggers a controlled shutdown sequence within 200
milliseconds.
Battery Efficiency vs Temperature
Efficiency (%)
87.5
90.0
92.5
95.0
0
10
20
30
40
50
Temperature (°C)

Figure 1. Efficiency of Orion A1 battery pack across temperature ranges. 0 dbfs:/Volumes/dbacademy/labuser15785576_1783348796/orion_docs/02_Orion_Battery_Management_System_Manual.pdf == page ==

2. Safety Guidelines

Field engineers must adhere to strict handling procedures when working on battery modules.
Each pack must be isolated from the main bus before removal. The Orion Diagnostic Interface (ODI) should be used to verify zero current before disengagement. Never expose the battery to open flame, solvents, or humidity above 90%. Only Orion-certified technicians are authorized to
perform module replacements.
Charge Retention Over Life Cycle
Capacity Retained (%)
0
200
400
600
800
1000
Charge Cycles

Figure 2. Retained charge capacity over lifecycle testing.

3. Technical Specifications

 Attribute Specification Nominal Voltage 52.8 V Capacity 18 Ah Energy Density 245 Wh/kg Cycle Life (80%) 1000 cycles Operating Temp Range 0–45 °C Protection Dual redundant current sensors + fuse isolation 

Routine inspection of the battery system is recommended every 500 operational hours. Key
diagnostic parameters include cell voltage deviation (<0.02V), temperature differential (<3°C),
and charge retention after standby. Logs generated by the BMS can be exported via the Orion
Diagnostics Utility for advanced analysis or remote monitoring.
End of Document 1 dbfs:/Volumes/dbacademy/labuser15785576_1783348796/orion_docs/01_Orion_A1_Product_Overview.pdf Orion Robotics

Orion A1 Product Overview
© 2025 Orion Robotics – Confidential
The Orion A1 humanoid service robot is the flagship model in the Orion Robotics product line.
Designed for adaptive environments, A1 combines modular hardware, precision motion control,
and deep learning-based sensory perception to perform tasks ranging from hospitality
assistance to warehouse logistics.
Since its initial release in 2023, Orion A1 has been deployed in over 40 countries. Each production cycle incorporates firmware refinements and mechanical upgrades based on telemetry data collected from field units. The 2025 A1 model introduces an improved gait controller and battery module rated for 10 hours of continuous operation. 2 dbfs:/Volumes/dbacademy/labuser15785576_17

## Summary and Next Steps

In this demo, we loaded parsed documents, applied both **LLM-powered semantic cleaning** and **fast plain text extraction**, and **chunked** the results for retrieval workflows. We then saved the chunked data to a Delta table, preparing it for embedding and integration with AI Search and LLM-powered applications.

**Key Takeaways:**
- Use LLM-powered semantic cleaning or fast concatenation to prepare text for chunking.
- Chunk text by page.
- Store chunked data in a Delta table for easy integration with AI Search and embedding pipelines.


&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>